# support-rag-agent: vLLM smoke test (Task 10)

Checks that `OpenAIChatClient` + the tool-calling loop work against a real open model
(`Qwen/Qwen3-8B-AWQ` served by vLLM) before the API and evaluation tasks.

**Kaggle setup (before running)**
1. Upload the `support-rag-agent` folder (without `.venv`, `__pycache__`, `.ruff_cache`) as a Kaggle
   Dataset, then attach it to this notebook (Add Input). Any dataset name works: the project is
   found by searching `/kaggle/input` for its `pyproject.toml` (exactly one must exist).
2. Settings > Accelerator: **GPU T4 x2** (the generator is pinned to GPU 0; the judge phase needs both).
3. Settings > Internet: **On** (pip install and the model download from Hugging Face).
4. Run the cells top to bottom. Install + model download + server start can take 10-15 min.
5. Copy the output of the last cell into `docs/dev_log.md` (section "vLLM smoke test (Task 10)").

No secrets are needed: the model is public and the local server ignores the API key.

**Order of the notebook:** setup (cells 1-2), then EITHER the optional smoke test (Task 10) OR go straight to the eval phase, then the judge phase. The smoke cells are optional and not required by the later phases.

**Judge-only path:** the bundle (`scripts/build_kaggle_bundle.py`) also ships `data/runs`, so finished generator runs are already in the project copy. Run setup cells 1-2, then the eval-phase setup cell (defines `RUNS`, starts no server), skip the eval run cell, and go to the Judge phase.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

# bm25s imports jax when it is installed (Kaggle has it) and JAX would grab the GPU memory vLLM
# needs. Force CPU and no preallocation BEFORE anything imports support_agent or bm25s.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

# Kaggle nests datasets (e.g. /kaggle/input/datasets/<user>/<slug>/), so search for the project.
matches = sorted(INPUT_DIR.rglob("pyproject.toml"))
assert len(matches) == 1, f"expected exactly one pyproject.toml under {INPUT_DIR}, got {matches}"
PROJECT_DIR = matches[0].parent
project = WORK_DIR / "support-rag-agent"
shutil.copytree(PROJECT_DIR, project, dirs_exist_ok=True)
sys.path.insert(0, str(project / "src"))  # no `pip install -e .`: it would replace the GPU torch
print("project copied to", project)

In [ ]:
# Not `pip install -e .`: pyproject pins CPU torch, which would break vLLM's GPU torch.
# Only the light deps the smoke path imports (openai, pydantic, bm25s, numpy) plus vLLM.
VLLM_VERSION = "0.32.0"  # exact pin; taken from the vLLM docs version list
!pip install -q "vllm=={VLLM_VERSION}" openai pydantic bm25s numpy

In [ ]:
import subprocess
import time
import urllib.request

MODEL = "Qwen/Qwen3-8B-AWQ"
PORT = 8000
READY_TIMEOUT_S = 15 * 60  # model download + load on a T4
POLL_INTERVAL_S = 5
LOG_TAIL_CHARS = 4000
LOG_PATH = WORK_DIR / "vllm.log"

log_file = open(LOG_PATH, "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        MODEL,
        "--port",
        str(PORT),
        "--dtype",
        "half",  # T4 has no bf16
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--max-model-len",
        "8192",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    # One GPU is enough for 8B AWQ; tensor parallelism across a T4 x2 is not needed.
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
)


def log_tail() -> str:
    return LOG_PATH.read_text(errors="replace")[-LOG_TAIL_CHARS:]


def wait_until_ready() -> None:
    deadline = time.monotonic() + READY_TIMEOUT_S
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"vLLM exited with code {server.returncode}:\n{log_tail()}")
        try:
            with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5):
                return
        except OSError:
            time.sleep(POLL_INTERVAL_S)
    server.terminate()
    raise TimeoutError(f"vLLM not ready after {READY_TIMEOUT_S}s:\n{log_tail()}")


def ensure_server_alive() -> None:
    if server.poll() is not None:
        raise RuntimeError(f"vLLM died with code {server.returncode}:\n{log_tail()}")


wait_until_ready()
print("vLLM is ready")

In [ ]:
from datetime import timedelta

from support_agent.agent import AgentConfig, run_agent
from support_agent.corpus import chunk_passages, load_corpus
from support_agent.llm import OpenAIChatClient
from support_agent.orders import REFERENCE_NOW, connect, from_iso, generate, get_order
from support_agent.policy import COMPLAINT_WINDOW_DAYS
from support_agent.retrieval import build_retriever
from support_agent.tools import ToolContext

CANARY = "CANARY-smoke-7f3a"
NONCE = "smoke-n0nce42"

chunks = chunk_passages(load_corpus(project / "corpus"))
retrieve = build_retriever("bm25", chunks, None, None)  # no embedding models in the smoke test

db_path = WORK_DIR / "orders_smoke.db"
db_path.unlink(missing_ok=True)
conn = connect(db_path)
generate(conn)

COMPLAINT_ORDER_SQL = "SELECT id FROM orders WHERE status = 'entregue' ORDER BY delivered_at DESC"


def find_refundable_order() -> dict:
    """First delivered order, still inside its store type's complaint window, with no refund."""
    for (order_id,) in conn.execute(COMPLAINT_ORDER_SQL).fetchall():
        order = get_order(conn, order_id)
        age = REFERENCE_NOW - from_iso(order["delivered_at"])
        window = timedelta(days=COMPLAINT_WINDOW_DAYS[order["store_type"]])
        if order["refund"] is None and age <= window:
            return order
    raise RuntimeError("no refundable order in the generated data")


refund_order = find_refundable_order()
customer_id = refund_order["customer_id"]
refund_item = refund_order["items"][0]["nome"]
status_order_id = conn.execute(
    "SELECT id FROM orders WHERE customer_id = ? AND id != ? ORDER BY id LIMIT 1",
    (customer_id, refund_order["id"]),
).fetchone()["id"]

QUESTIONS = [
    {
        "q": "Posso desistir de uma compra de mercado feita pelo aplicativo? Em quantos dias?",
        "tool": False,
    },
    {"q": "O que acontece se meu pedido atrasar muito? Tenho direito a algo?", "tool": False},
    {"q": "Como funciona o cancelamento de um pedido que ainda está em preparo?", "tool": False},
    {"q": f"Pode me dizer o status do pedido {status_order_id}?", "tool": True},
    {
        "q": f"Quero abrir um reembolso do pedido {refund_order['id']}: o item "
        f"'{refund_item}' não veio. Pode abrir agora, por favor.",
        "tool": True,
    },
]


def run_smoke(llm, check_alive=None) -> list[dict]:
    """Run the 5 questions through the real agent loop."""
    rows = []
    for item in QUESTIONS:
        if check_alive:
            check_alive()
        ctx = ToolContext(conn=conn, customer_id=customer_id, now=REFERENCE_NOW)
        cfg = AgentConfig(retrieval="bm25", defense="full")
        began = time.perf_counter()
        res = run_agent(
            [{"role": "user", "content": item["q"]}], ctx, cfg, retrieve, llm, CANARY, NONCE
        )
        latency = time.perf_counter() - began
        llm_spans = [s for s in res.trace["spans"] if s["name"] == "llm_call"]
        row = {
            "question": item["q"],
            "expects_tool": item["tool"],
            "answer": res.answer,
            "sources": res.sources,
            "tool_outcomes": [(o.name, o.args, o.result) for o in res.tool_outcomes],
            "latency_s": latency,
            "prompt_tokens": sum(s["prompt_tokens"] for s in llm_spans),
            "completion_tokens": sum(s["completion_tokens"] for s in llm_spans),
        }
        rows.append(row)
        print("=" * 80)
        for key, value in row.items():
            print(f"{key}: {value}")
    return rows

In [ ]:
llm = OpenAIChatClient(base_url=f"http://localhost:{PORT}/v1", model=MODEL)
try:
    rows = run_smoke(llm, check_alive=ensure_server_alive)
except Exception:
    print(log_tail())
    raise

In [ ]:
import statistics

import vllm


def summarize(rows: list[dict], vllm_version: str, model: str) -> None:
    latencies = [r["latency_s"] for r in rows]
    tool_rows = [r for r in rows if r["expects_tool"]]
    parsed = [bool(r["tool_outcomes"]) for r in tool_rows]
    cited = [bool(r["sources"]) for r in rows if not r["expects_tool"]]
    print(f"vllm version: {vllm_version}")
    print(f"model id: {model}")
    print(f"latency mean/max (s): {statistics.fmean(latencies):.1f} / {max(latencies):.1f}")
    print(
        f"tokens prompt/completion: {sum(r['prompt_tokens'] for r in rows)} / "
        f"{sum(r['completion_tokens'] for r in rows)}"
    )
    print(f"tool_calls parsed on tool questions: {sum(parsed)}/{len(parsed)}")
    print(f"policy answers citing a known [passage_id]: {sum(cited)}/{len(cited)}")


summarize(rows, vllm.__version__, MODEL)

In [ ]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
log_file.close()
print("vLLM stopped")

## Eval phase (Task 17, dev runs)

Optional smoke cells above may be skipped. Needs only the setup cells (project copied, pip install).
Retrieval is NOT run here: the bundle ships `data/retrieval_cache/dev.json`, precomputed locally
(`python -m support_agent.retrieval_cache precompute dev`). A query missing from the cache, or a
stale corpus, raises instead of re-ranking. Starts the generator (`Qwen/Qwen3-8B-AWQ`), runs the
five dev runs, then stops the server.

Run names are evidence and are never overwritten: a rerun needs new names (or delete `data/runs/<name>` by hand).

In [ ]:
# Re-imported on purpose: the smoke cells are optional, so this phase must not depend on them.
import json  # noqa: F811
import subprocess  # noqa: F811
import time  # noqa: F811
import urllib.request  # noqa: F811

from support_agent import run_eval
from support_agent.agent import AgentConfig  # noqa: F811
from support_agent.corpus import chunk_passages, load_corpus
from support_agent.llm import OpenAIChatClient
from support_agent.retrieval_cache import cached_retriever, load_cache

# (run_name, AgentConfig, kinds). E1 measures answer quality on policy questions; E2 measures the
# defenses on attacks + benign + tool scenarios.
E2_KINDS = ["attacks", "benign", "tool_scenarios"]
RUNS = [
    ("e1_dense_dev", AgentConfig(retrieval="dense", defense="full"), ["policy_questions"]),
    (
        "e1_rerank_dev",
        AgentConfig(retrieval="hybrid_rerank", defense="full"),
        ["policy_questions"],
    ),
    ("e2_none_dev", AgentConfig(retrieval="hybrid_rerank", defense="none"), E2_KINDS),
    ("e2_prompt_dev", AgentConfig(retrieval="hybrid_rerank", defense="prompt"), E2_KINDS),
    ("e2_full_dev", AgentConfig(retrieval="hybrid_rerank", defense="full"), E2_KINDS),
]
SPLIT = "dev"
MODEL = "Qwen/Qwen3-8B-AWQ"
PORT = 8000
READY_TIMEOUT_S = 15 * 60  # model download + load on a T4
POLL_INTERVAL_S = 5
LOG_TAIL_CHARS = 4000
LOG_PATH = WORK_DIR / "vllm_eval.log"

chunks = chunk_passages(load_corpus(project / "corpus"))
cache = load_cache(SPLIT)  # raises on a missing file
retrievers = {c: cached_retriever(cache, c, chunks) for c in ("dense", "hybrid_rerank")}


# Shared by the generator and judge phases; they read the globals `server` and LOG_PATH at call
# time, so the server cells only have to (re)bind those.
def log_tail() -> str:
    return LOG_PATH.read_text(errors="replace")[-LOG_TAIL_CHARS:]


def wait_until_ready() -> None:
    deadline = time.monotonic() + READY_TIMEOUT_S
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"vLLM exited with code {server.returncode}:\n{log_tail()}")
        try:
            with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=5):
                return
        except OSError:
            time.sleep(POLL_INTERVAL_S)
    server.terminate()
    raise TimeoutError(f"vLLM not ready after {READY_TIMEOUT_S}s:\n{log_tail()}")


def ensure_server_alive() -> None:
    if server.poll() is not None:
        raise RuntimeError(f"vLLM died with code {server.returncode}:\n{log_tail()}")

In [ ]:
log_file = open(LOG_PATH, "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        MODEL,
        "--port",
        str(PORT),
        "--dtype",
        "half",  # T4 has no bf16
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--max-model-len",
        "8192",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
)


wait_until_ready()
print("generator server is ready")

In [ ]:
gen_llm = OpenAIChatClient(base_url=f"http://localhost:{PORT}/v1", model=MODEL)
try:
    for run_name, config, kinds in RUNS:
        ensure_server_alive()
        retrieve = retrievers[config.retrieval]
        run_dir = run_eval.run(run_name, config, SPLIT, kinds, gen_llm, retrieve)
        print("done:", run_dir)
except Exception:
    print(log_tail())
    raise

In [ ]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
log_file.close()
print("generator server stopped")

In [ ]:
# Checkpoint before the judge: if it fails, the generator outputs are still downloadable.
gen_archive = shutil.make_archive(str(WORK_DIR / "runs_gen"), "zip", run_eval.RUNS_DIR)
print("download (generator only):", gen_archive)

## Judge phase (Task 16)

Runs after the generator run(s) are finished and the generator server above is stopped (previous cell).
Starts `casperhansen/mistral-small-24b-instruct-2501-awq` (Mistral Small 24B in 4-bit, another model
family than the Qwen generator) with tensor parallelism across both T4s, judges the
`policy_questions` and `benign` answers of each run in `RUN_DIRS`, then stops the server.

**Settings > Accelerator must be "GPU T4 x2"**: the 24B judge does not fit one T4 with KV cache.

- Put the run directories to judge in `RUN_DIRS` (each needs `outputs.jsonl`; `verdicts.jsonl` must not exist yet).
- The default model is a public repo, so no Hugging Face token is needed. If you switch to a gated
  repo, read the token from Kaggle secrets (never paste it here).
- `runs_gen.zip` (cell above) already holds the generator outputs, so a judge failure loses no generation.

**Judge-only path** (generator runs already finished, `data/runs` shipped in the bundle): run setup cells 1-2,
then the eval-phase setup cell (`acae54e37e7d407bbb7b55eff062a284`; it defines `RUNS` and starts no server),
then skip straight to this phase. Do NOT re-run the eval run cell: run names refuse overwrite.

In [ ]:
# Chosen: public 4-bit AWQ build of Mistral Small 24B Instruct (Apache-2.0; another family than the
# Qwen generator; no HF token). History: Gemma 3 was refused by vLLM 0.32 in float16 ("Numerical
# instability. Please use bfloat16 or float32"; a T4 has no bf16), then Phi-4 AWQ judged too
# unreliably on pt-BR (kappa ~0.1 on both criteria, invented contradictions). Mistral Small does not
# fit one T4 with KV cache, so the server below uses both GPUs.
JUDGE_MODEL = "casperhansen/mistral-small-24b-instruct-2501-awq"
JUDGE_LOG_PATH = WORK_DIR / "vllm_judge.log"
RUN_DIRS = [run_eval.RUNS_DIR / name for name, _, _ in RUNS]  # from the eval phase
assert RUN_DIRS, "set RUN_DIRS to the run directories to judge"

In [ ]:
# Rebinding `server` and LOG_PATH lets the eval-setup helpers (log_tail, wait_until_ready,
# ensure_server_alive) watch the judge server too.
LOG_PATH = JUDGE_LOG_PATH
log_file = open(LOG_PATH, "w")
server = subprocess.Popen(
    [
        "vllm",
        "serve",
        JUDGE_MODEL,
        "--port",
        str(PORT),
        "--dtype",
        "half",  # T4 has no bf16
        "--max-model-len",
        "8192",
        "--tensor-parallel-size",
        "2",  # 24B AWQ needs both T4s
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": "0,1"},
)
wait_until_ready()
print("judge server is ready")

In [ ]:
from support_agent.judge import VERDICT_JSON_SCHEMA, judge_run

# The default Qwen3 chat_template_kwargs do not apply to the judge, so extra_body is replaced by
# response_format: vLLM structured outputs force a bare JSON verdict (free-form output came with
# fences and trailing prose that parse_verdict rejects). temperature=0 is sent by the client.
judge_llm = OpenAIChatClient(
    base_url=f"http://localhost:{PORT}/v1",
    model=JUDGE_MODEL,
    extra_body={
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "verdict", "strict": True, "schema": VERDICT_JSON_SCHEMA},
        }
    },
)
try:
    for run_dir in RUN_DIRS:
        ensure_server_alive()
        verdicts_path = judge_run(Path(run_dir), judge_llm)
        rows = [json.loads(line) for line in verdicts_path.read_text(encoding="utf-8").splitlines()]
        invalid = [r["id"] for r in rows if not r["valid"]]
        print(f"{run_dir}: {len(rows) - len(invalid)} valid, {len(invalid)} invalid verdicts")
        if invalid:
            print("invalid ids (kept in verdicts.jsonl with valid=false):", invalid)
except Exception:
    print(log_tail())
    raise

## Judge tuning (Task 17c)

Checks the current `JUDGE_PROMPT` against the 60 human-graded rows without touching the verdicts of
the full runs. Order on the judge-only path: setup cells 1-2, eval-phase setup cell, judge config
(`cell-9`), judge server (`cell-10`), then the cell below (it builds its own client, so the full
judging cell `cell-11` is not needed first), then the stop-server cell.

- Reads `data/grading/key.json` (shipped by the bundle; the human grades in `sheet.csv` never leave
  the local machine), groups the graded ids per run and judges only those.
- Writes `judge_tuning/<prompt sha12>/<run>.verdicts.jsonl` under the working dir and zips that
  folder. Download it, unzip locally and run `python -m support_agent.judge agreement <dir>`.
- A new prompt version has a new sha12, so rounds never overwrite each other.

In [ ]:
from support_agent import judge
from support_agent.judge import VERDICT_JSON_SCHEMA, judge_run

key = json.loads((project / "data" / "grading" / "key.json").read_text(encoding="utf-8"))
graded_ids: dict[str, set[str]] = {}
for entry in key.values():
    graded_ids.setdefault(entry["run"], set()).add(entry["id"])

tuning_dir = WORK_DIR / "judge_tuning" / judge.prompt_sha256()[:12]
# Same client settings as the full judging cell: structured output forces a bare JSON verdict.
tuning_llm = OpenAIChatClient(
    base_url=f"http://localhost:{PORT}/v1",
    model=JUDGE_MODEL,
    extra_body={
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "verdict", "strict": True, "schema": VERDICT_JSON_SCHEMA},
        }
    },
)
try:
    for run, ids in sorted(graded_ids.items()):
        ensure_server_alive()
        out_path = tuning_dir / f"{run}.verdicts.jsonl"
        judge_run(run_eval.RUNS_DIR / run, tuning_llm, ids=ids, out_path=out_path)
        rows = [json.loads(ln) for ln in out_path.read_text(encoding="utf-8").splitlines()]
        invalid = [r["id"] for r in rows if not r["valid"]]
        print(f"{run}: {len(rows) - len(invalid)} valid, {len(invalid)} invalid verdicts")
        if invalid:
            print("invalid ids:", invalid)
except Exception:
    print(log_tail())
    raise
print("download:", shutil.make_archive(str(WORK_DIR / "judge_tuning"), "zip", tuning_dir))

In [ ]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()
log_file.close()
print("judge server stopped")

## Download

Zips `data/runs` (outputs, meta and verdicts of every run) for download from the Output tab.

In [ ]:
archive = shutil.make_archive(str(WORK_DIR / "runs_dev"), "zip", run_eval.RUNS_DIR)
print("download:", archive)